In [ ]:
HELD_OUT=[4,5,6]  # <-- SADECE BU SATIRI DEGISTIR (hesap basina farkli denekler)
# ============================================================
import os
os.system("pip install snntorch -q")
import glob, re, warnings, random, gc, json
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, confusion_matrix
from tqdm import tqdm
import snntorch as snn
from snntorch import surrogate
warnings.filterwarnings('ignore')
S=42; random.seed(S); np.random.seed(S); torch.manual_seed(S); torch.cuda.manual_seed_all(S)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cihaz:",device,"| Disarida birakilan denekler (LOSO):",HELD_OUT)

local_semg_dir, local_fatigue_dir = "/content/sEMG_data", "/content/fatigue_labels"
if not glob.glob(os.path.join(local_semg_dir,"**/*.csv"),recursive=True):
    import requests, zipfile
    print("Zenodo'dan indiriliyor...")
    os.system("apt-get install -y aria2 > /dev/null 2>&1")
    r=requests.get("https://zenodo.org/api/records/13937111").json()
    url=[f for f in r["files"] if f["key"]=="sEMG_data.zip"][0]["links"]["self"]
    os.system(f'aria2c -x 16 -s 16 -c -d /content -o sEMG_data.zip "{url}" > /dev/null 2>&1')
    os.makedirs(local_semg_dir,exist_ok=True)
    with zipfile.ZipFile("/content/sEMG_data.zip") as z: z.extractall(local_semg_dir)
    for nz in glob.glob(os.path.join(local_semg_dir,"**/*.zip"),recursive=True):
        with zipfile.ZipFile(nz) as z: z.extractall(os.path.dirname(nz))
        os.remove(nz)
    os.remove("/content/sEMG_data.zip")
    fz=[f for f in r["files"] if "fatigue" in f["key"].lower()][0]["links"]["self"]
    os.system(f'aria2c -x 16 -s 16 -c -d /content -o fat.zip "{fz}" > /dev/null 2>&1')
    os.makedirs(local_fatigue_dir,exist_ok=True)
    with zipfile.ZipFile("/content/fat.zip") as z: z.extractall("/content/_ft")
    for nz in glob.glob("/content/_ft/**/*.zip",recursive=True):
        with zipfile.ZipFile(nz) as z: z.extractall(local_fatigue_dir)
    os.remove("/content/fat.zip")
print("Veri hazir.")

FS=1259; WINDOW=int(5.0*FS); STEP=WINDOW//2
semg_files=sorted([f for f in glob.glob(os.path.join(local_semg_dir,"**/*.csv"),recursive=True) if "mvc" not in f.lower()])
label_files=sorted(glob.glob(os.path.join(local_fatigue_dir,"**/*.csv"),recursive=True))
def est(fp):
    s=re.search(r'subject_(\d+)',os.path.dirname(fp).lower()); t=re.search(r'trial_(\d+)',os.path.basename(fp).lower())
    return (int(s.group(1)),int(t.group(1))) if s and t else (None,None)
semg_dict={est(f):f for f in semg_files if est(f)[0]}; label_dict={est(f):f for f in label_files if est(f)[0]}
keys=sorted(set(semg_dict)&set(label_dict))
def load_raw(sp,lp):
    df=pd.read_csv(sp); cols=[c for c in df.columns if '[V]' in c and not c.strip().startswith('X')]
    if not cols: return None,None
    raw=np.nan_to_num(df[cols].values.astype(np.float32),nan=0,posinf=0,neginf=0)
    tc=df.iloc[:,0].values; dl=pd.read_csv(lp); lt,ll=dl.iloc[:,0].values,dl.iloc[:,1].values
    idx=np.clip(np.searchsorted(lt,tc),0,len(ll)-1); al=ll[idx].astype(int)
    m,s=raw.mean(0,keepdims=True),raw.std(0,keepdims=True); s[s<1e-6]=1.0
    return np.nan_to_num((raw-m)/s,nan=0,posinf=0,neginf=0), al
window_fn=torch.hann_window(64); spec_list=[]; ys=[]; subjs=[]
print("Yukleme + STFT (bir kere)...")
for (sj,tr) in tqdm(keys):
    d,lb=load_raw(semg_dict[(sj,tr)],label_dict[(sj,tr)])
    if d is None or len(d)<=WINDOW: continue
    starts=list(range(0,len(d)-WINDOW,STEP)); n=len(starts)
    ch=np.stack([d[i:i+WINDOW] for i in starts]).astype(np.float32)
    lls=[int(np.bincount(lb[i:i+WINDOW],minlength=3).argmax()) for i in starts]
    xt=torch.from_numpy(ch).transpose(1,2).reshape(n*4,WINDOW)
    st=torch.abs(torch.stft(xt,n_fft=64,hop_length=32,window=window_fn,return_complex=True))
    st=st.view(n,4,33,-1).permute(0,3,1,2).contiguous()
    spec_list.append(st); ys+=lls; subjs+=[sj]*n; del d,ch,xt,st
spec_all=torch.cat(spec_list,0); del spec_list; gc.collect()
spec_all=torch.nan_to_num(spec_all,nan=0,posinf=0,neginf=0); spec_all=spec_all/(spec_all.max()+1e-8)
raw_y=np.array(ys); subj_arr=np.array(subjs)
keep=np.isin(raw_y,[0,2]); kt=torch.from_numpy(keep)
spec_all=spec_all[kt]; raw_y=raw_y[keep]; subj_arr=subj_arr[keep]
y_all=torch.tensor((raw_y==2).astype(int),dtype=torch.long)
thr=0.5*spec_all.std(); dx=(spec_all[:,1:]-spec_all[:,:-1]).abs(); sp=(dx>=thr); del dx; gc.collect()
X_all=torch.cat([torch.zeros_like(spec_all[:,:1],dtype=torch.bool),sp],dim=1); del spec_all,sp; gc.collect()
print("Toplam pencere:",len(y_all),"| Toplam denek:",len(np.unique(subj_arr)))

class ConvSNN(nn.Module):
    def __init__(s,n_out=2,beta=0.85,p=0.3):
        super().__init__(); sg=surrogate.fast_sigmoid(slope=25)
        s.conv1=nn.Conv1d(4,16,5,padding=2); s.bn1=nn.BatchNorm1d(16)
        s.lif1=snn.Leaky(beta=beta,threshold=0.3,spike_grad=sg,learn_beta=True); s.drop=nn.Dropout(p)
        s.fc1=nn.Linear(16*33,256); s.lif2=snn.Leaky(beta=beta,threshold=0.3,spike_grad=sg,learn_beta=True)
        s.fc2=nn.Linear(256,n_out); s.lif3=snn.Leaky(beta=beta,threshold=0.3,spike_grad=sg,learn_beta=True)
    def forward(s,x):
        x=x.permute(1,0,2,3); m1,m2,m3=s.lif1.init_leaky(),s.lif2.init_leaky(),s.lif3.init_leaky(); mem=[]; ss=0.0
        for t in range(x.size(0)):
            a1,m1=s.lif1(s.bn1(s.conv1(x[t])),m1); f=s.drop(a1.view(x.size(1),-1))
            a2,m2=s.lif2(s.fc1(f),m2); a3,m3=s.lif3(s.fc2(a2),m3); mem.append(m3); ss=ss+a1.mean()+a2.mean()
        return torch.stack(mem), ss/(2*x.size(0))

class ConvANN(nn.Module):
    def __init__(s,n_out=2,p=0.3):
        super().__init__(); s.conv1=nn.Conv1d(4,16,5,padding=2); s.bn1=nn.BatchNorm1d(16); s.relu=nn.ReLU(); s.drop=nn.Dropout(p)
        s.fc1=nn.Linear(16*33,256); s.fc2=nn.Linear(256,n_out)
    def forward(s,x):
        x=x.permute(1,0,2,3); outs=[]
        for t in range(x.size(0)):
            c=s.relu(s.bn1(s.conv1(x[t]))); f=s.drop(c.view(x.size(1),-1)); outs.append(s.fc2(s.relu(s.fc1(f))))
        return torch.stack(outs), torch.tensor(0.0,device=x.device)

def train_eval(model_type, tri, vai, epochs=25, patience=8):
    Xtr,Xva=X_all[tri],X_all[vai]; ytr,yva=y_all[tri],y_all[vai]
    tc=np.bincount(ytr.numpy(),minlength=2); cw=torch.tensor(len(ytr)/(2*tc),dtype=torch.float32).to(device)
    tl=DataLoader(TensorDataset(Xtr,ytr),batch_size=128,shuffle=True,drop_last=True)
    vl=DataLoader(TensorDataset(Xva,yva),batch_size=128,shuffle=False)
    model=(ConvSNN(p=0.3) if model_type=="snn" else ConvANN(p=0.3)).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=2e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs); crit=nn.CrossEntropyLoss(weight=cw)
    best_f1,best_ep,best_pred=0,0,None
    for ep in range(epochs):
        model.train()
        for xb,yb in tl:
            xb=xb.to(device).float(); yb=yb.to(device)
            out,spk=model(xb); logits=out.mean(0); loss=crit(logits,yb)+(0.5*spk if model_type=="snn" else 0.0)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        sch.step(); model.eval(); vp=[]
        with torch.no_grad():
            for xb,_ in vl: out,_=model(xb.to(device).float()); vp.append(out.mean(0).argmax(1).cpu())
        vp=torch.cat(vp).numpy(); vf=f1_score(yva.numpy(),vp,average='macro')*100
        if vf>best_f1: best_f1,best_ep,best_pred=vf,ep,vp
        if ep-best_ep>=patience: break
    return best_pred

snn_cm=np.zeros((2,2),dtype=int); cnn_cm=np.zeros((2,2),dtype=int)
mcnemar={"both_correct":0,"snn_only_correct":0,"cnn_only_correct":0,"both_wrong":0}

for held in HELD_OUT:
    print(f"\n===== LOSO fold: denek {held} disarida =====")
    tri=np.where(subj_arr!=held)[0]; vai=np.where(subj_arr==held)[0]
    if len(vai)==0: print("  (bu denek veri setinde yok, atlaniyor)"); continue
    yv=y_all[vai].numpy()
    snn_pred=train_eval("snn",tri,vai)
    cnn_pred=train_eval("cnn",tri,vai)
    snn_cm+=confusion_matrix(yv,snn_pred,labels=[0,1])
    cnn_cm+=confusion_matrix(yv,cnn_pred,labels=[0,1])
    for t,sp_,cp_ in zip(yv,snn_pred,cnn_pred):
        sc=(sp_==t); cc=(cp_==t)
        if sc and cc: mcnemar["both_correct"]+=1
        elif sc and not cc: mcnemar["snn_only_correct"]+=1
        elif cc and not sc: mcnemar["cnn_only_correct"]+=1
        else: mcnemar["both_wrong"]+=1
    print(f"  Fold ozet -> SNN acc: {(snn_pred==yv).mean()*100:.1f}% | CNN acc: {(cnn_pred==yv).mean()*100:.1f}%")

result={"held_out_subjects":HELD_OUT,"snn_confusion":snn_cm.tolist(),"cnn_confusion":cnn_cm.tolist(),"mcnemar_table":mcnemar}
print("\n=== SONUC JSON (bana yapistir) ===")
print(json.dumps(result,indent=2))

Cihaz: cuda | Disarida birakilan denekler (LOSO): [4, 5, 6]
Zenodo'dan indiriliyor...
Veri hazir.
Yukleme + STFT (bir kere)...


100%|██████████| 156/156 [02:24<00:00,  1.08it/s]


Toplam pencere: 12370 | Toplam denek: 13

===== LOSO fold: denek 4 disarida =====
  Fold ozet -> SNN acc: 80.9% | CNN acc: 83.8%

===== LOSO fold: denek 5 disarida =====
  Fold ozet -> SNN acc: 82.7% | CNN acc: 78.3%

===== LOSO fold: denek 6 disarida =====
  Fold ozet -> SNN acc: 85.9% | CNN acc: 86.6%

=== SONUC JSON (bana yapistir) ===
{
  "held_out_subjects": [
    4,
    5,
    6
  ],
  "snn_confusion": [
    [
      1130,
      132
    ],
    [
      223,
      654
    ]
  ],
  "cnn_confusion": [
    [
      1148,
      114
    ],
    [
      238,
      639
    ]
  ],
  "mcnemar_table": {
    "both_correct": 1652,
    "snn_only_correct": 132,
    "cnn_only_correct": 135,
    "both_wrong": 220
  }
}
